# 37-Seoul. 서울코퍼스에서 모음충돌회피 발음 검색

37번 사전검색 결과(모음조화/모음충돌회피 후보)가 서울코퍼스에서 실제로 어떻게 발음되는지 확인

## 핵심 기능
1. 37번 어간 리스트 → 서울 pWord에서 활용형(어간+어미) 매칭
2. ortho vs prono 비교: 탈락 / 활음화 / 활음삽입 중 어떤 전략이 실현되는지
3. 화자 변수(성별/연령)별 변이 통계

## 입력
- 37번 결과: `search_results/vowel_harmony_collision_all_*.csv`
- 서울 pWord enriched: `04_seoul_pword_enriched.csv` (242MB, 231,632 pWord)

## 출력
- `37_vowel_harmony_collision/search_results/seoul_vowel_collision_*.csv`

## 1. 환경 설정

In [1]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/DATA_2026'
except ImportError:
    PROJECT_ROOT = os.path.dirname(os.getcwd())
    print(f'Local mode: {PROJECT_ROOT}')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import re

# 경로 설정
SEARCH_37 = f'{PROJECT_ROOT}/37_vowel_harmony_collision/search_results'
SEOUL_PWORD = f'{PROJECT_ROOT}/00_raw_data/03_seoul_corpus/02_csv/04_seoul_pword_enriched.csv'
RESULT_DIR = f'{PROJECT_ROOT}/37_vowel_harmony_collision/search_results'
os.makedirs(RESULT_DIR, exist_ok=True)

print(f'37번 결과: {SEARCH_37}')
print(f'서울 pWord: {SEOUL_PWORD}')

37번 결과: /content/drive/MyDrive/DATA_2026/37_vowel_harmony_collision/search_results
서울 pWord: /content/drive/MyDrive/DATA_2026/00_raw_data/03_seoul_corpus/02_csv/04_seoul_pword_enriched.csv


## 2. 37번 결과 로드 — 어간 리스트 추출

In [3]:
# 37번 결과 로드
all_files = sorted(Path(SEARCH_37).glob('vowel_harmony_collision_all_*.csv'))
if not all_files:
    raise FileNotFoundError('37번 결과 CSV가 없습니다')

latest = all_files[-1]
print(f'사용 파일: {latest.name}')

df37 = pd.read_csv(latest, encoding='utf-8-sig')
print(f'37번 결과: {len(df37):,}행')
print(f'environment: {df37["environment"].value_counts().to_dict()}')

# 어간 → 정보 매핑 딕셔너리 (어간을 키로, 관련 정보를 값으로)
stem_info = {}
for _, row in df37.iterrows():
    stem = row['stem']
    if pd.isna(stem):
        continue
    stem_info[stem] = {
        'word': row['word'],
        'environment': row['environment'],
        'collision_type': row.get('collision_type', ''),
        'stem_final_vowel': row.get('stem_final_vowel_name', ''),
        'vowel_polarity': row.get('vowel_polarity', ''),
        'suffix_type': row.get('suffix_type', ''),
    }

stems = list(stem_info.keys())
print(f'고유 어간: {len(stems):,}개')

사용 파일: vowel_harmony_collision_all_20260312_071836.csv
37번 결과: 93,413행
environment: {'vowel_collision': 87271, 'vowel_harmony': 6142}
고유 어간: 59,597개


## 3. 서울 pWord 로드

In [ ]:
# 서울 pWord enriched 로드 (242MB)
print('서울 pWord 로딩...')
df_seoul = pd.read_csv(SEOUL_PWORD, encoding='utf-8-sig', low_memory=False)
print(f'서울 pWord: {len(df_seoul):,}행')
print(f'컬럼: {list(df_seoul.columns)}')
print(f'\n예시:')
df_seoul.head(3)

서울 pWord 로딩...


## 4. 어간 매칭 — pWord에서 활용형 검색

서울 코퍼스 pWord_ortho에서 37번 어간으로 시작하는 활용형을 찾는다.
- 예: 어간 '보' → '보아', '봐', '보고', '보니' 등
- 예: 어간 '가' → '가아'(→갔), '가서', '가니' 등

모음충돌 환경에 해당하는 어미(-아/-어)가 결합된 경우를 우선 추출.

In [ ]:
# morphs 컬럼에서 어간 매칭 (형태소 분석 결과 활용)
# morphs 형식: '보/VV+아/EC' 또는 '가/VV+아서/EC' 등

def extract_stem_from_morphs(morphs_str):
    """morphs 컬럼에서 용언 어간 추출"""
    if pd.isna(morphs_str):
        return None
    parts = str(morphs_str).split('+')
    for part in parts:
        if '/' in part:
            morph, pos = part.rsplit('/', 1)
            if pos in ('VV', 'VA', 'VX'):
                return morph
    return None

def extract_suffix_from_morphs(morphs_str):
    """morphs 컬럼에서 용언 다음 어미 추출"""
    if pd.isna(morphs_str):
        return None
    parts = str(morphs_str).split('+')
    found_verb = False
    for part in parts:
        if '/' in part:
            morph, pos = part.rsplit('/', 1)
            if found_verb and pos in ('EC', 'EF', 'EP', 'ETN', 'ETM'):
                return morph
            if pos in ('VV', 'VA', 'VX'):
                found_verb = True
    return None

# 어간 추출
print('어간 추출 중...')
df_seoul['verb_stem'] = df_seoul['morphs'].apply(extract_stem_from_morphs)
df_seoul['verb_suffix'] = df_seoul['morphs'].apply(extract_suffix_from_morphs)

# 용언이 있는 pWord만
df_verb = df_seoul[df_seoul['verb_stem'].notna()].copy()
print(f'용언 포함 pWord: {len(df_verb):,}행 / {len(df_seoul):,}행')
print(f'고유 어간: {df_verb["verb_stem"].nunique():,}개')

In [ ]:
# 37번 어간과 매칭
stem_set = set(stems)
df_matched = df_verb[df_verb['verb_stem'].isin(stem_set)].copy()
print(f'37번 어간 매칭: {len(df_matched):,}행')
print(f'매칭된 고유 어간: {df_matched["verb_stem"].nunique():,}개 / {len(stem_set):,}개')

# 매칭 안 된 어간
matched_stems = set(df_matched['verb_stem'].unique())
unmatched_stems = stem_set - matched_stems
print(f'매칭 안 된 어간: {len(unmatched_stems):,}개')
if len(unmatched_stems) > 0:
    print(f'  예시: {list(unmatched_stems)[:10]}')

In [ ]:
# 37번 정보 병합
df_matched['dict_word'] = df_matched['verb_stem'].map(lambda s: stem_info.get(s, {}).get('word', ''))
df_matched['dict_environment'] = df_matched['verb_stem'].map(lambda s: stem_info.get(s, {}).get('environment', ''))
df_matched['dict_collision_type'] = df_matched['verb_stem'].map(lambda s: stem_info.get(s, {}).get('collision_type', ''))
df_matched['dict_stem_final_vowel'] = df_matched['verb_stem'].map(lambda s: stem_info.get(s, {}).get('stem_final_vowel', ''))
df_matched['dict_vowel_polarity'] = df_matched['verb_stem'].map(lambda s: stem_info.get(s, {}).get('vowel_polarity', ''))

print('37번 정보 병합 완료')
print(df_matched[['pWord_ortho', 'verb_stem', 'verb_suffix', 'dict_word',
                   'dict_environment', 'dict_collision_type']].head(10).to_string())

## 5. 모음충돌 환경 필터 — -아/-어 어미 결합

In [ ]:
# 모음충돌 관련 어미: -아, -어, -아서, -어서, -았-, -었- 등
collision_suffixes = {'아', '어', '아서', '어서', '아도', '어도', '아야', '어야',
                      '았', '었', '아요', '어요', '아라', '어라', '아지', '어지'}

df_collision = df_matched[df_matched['verb_suffix'].isin(collision_suffixes)].copy()
print(f'모음충돌 어미 결합: {len(df_collision):,}행 / {len(df_matched):,}행')
print(f'\n어미 분포:')
print(df_collision['verb_suffix'].value_counts().head(10))

## 6. ortho vs prono 비교 — 충돌회피 전략 감지

모음충돌 환경에서 실제 발음이 어떤 전략을 사용하는지:
- **탈락**: 모음 하나가 사라짐 (예: 보+아 → [바])
- **활음화**: 모음이 활음(w/j)으로 변환 (예: 보+아 → [봐], 가+어 → [겨])
- **활음삽입**: 사이에 활음 삽입 (예: 가+아 → [가야])
- **동일**: 철자=발음 (변이 없음)

In [ ]:
def detect_collision_strategy(ortho_roman, prono_roman, ortho, prono):
    """
    ortho vs prono 로마자를 비교하여 모음충돌회피 전략 감지

    Returns: 'deletion' / 'glide_formation' / 'glide_insertion' / 'same' / 'unknown'
    """
    if pd.isna(prono_roman) or pd.isna(ortho_roman):
        return 'no_prono'

    o_r = str(ortho_roman).strip()
    p_r = str(prono_roman).strip()
    o_k = str(ortho).strip()
    p_k = str(prono).strip()

    if o_r == p_r:
        return 'same'

    # 음절 수 비교
    o_syls = o_r.split('-')
    p_syls = p_r.split('-')

    if len(p_syls) < len(o_syls):
        # 음절 수 감소 → 탈락 또는 활음화(축약)
        # 활음화: w/j가 발음에 포함
        if any(c in p_r.upper() for c in ['WA', 'WEO', 'WE', 'WI', 'YEO', 'YA']):
            return 'glide_formation'
        return 'deletion'
    elif len(p_syls) > len(o_syls):
        # 음절 수 증가 → 활음삽입 가능성
        return 'glide_insertion'
    else:
        # 음절 수 동일하지만 발음이 다름
        if o_k != p_k:
            return 'other_change'
        return 'same'

# 전략 감지 적용
if len(df_collision) > 0:
    df_collision['strategy'] = df_collision.apply(
        lambda row: detect_collision_strategy(
            row.get('pWord_ortho_roman'), row.get('pWord_prono_roman'),
            row.get('pWord_ortho'), row.get('pWord_prono')
        ), axis=1
    )
    print('=== 모음충돌회피 전략 분포 ===')
    print(df_collision['strategy'].value_counts())
    print(f'\n전략별 예시:')
    for strat in df_collision['strategy'].unique():
        sub = df_collision[df_collision['strategy'] == strat]
        print(f'\n--- {strat} ({len(sub):,}건) ---')
        print(sub[['pWord_ortho', 'pWord_prono', 'verb_stem', 'verb_suffix']].head(5).to_string())
else:
    print('모음충돌 어미 결합 데이터 없음')

## 7. 화자 변수별 변이 통계

In [ ]:
if len(df_collision) > 0 and 'strategy' in df_collision.columns:
    # 성별
    if 'spk_gender' in df_collision.columns:
        print('=== 성별 × 전략 ===')
        ct_gender = pd.crosstab(df_collision['spk_gender'], df_collision['strategy'], margins=True)
        print(ct_gender)
        print()
        # 비율
        ct_gender_pct = pd.crosstab(df_collision['spk_gender'], df_collision['strategy'], normalize='index').round(3)
        print(ct_gender_pct)

    # 연령
    if 'spk_age' in df_collision.columns:
        print('\n=== 연령 × 전략 ===')
        ct_age = pd.crosstab(df_collision['spk_age'], df_collision['strategy'], margins=True)
        print(ct_age)
        print()
        ct_age_pct = pd.crosstab(df_collision['spk_age'], df_collision['strategy'], normalize='index').round(3)
        print(ct_age_pct)

In [ ]:
# 어간말모음별 전략 분포
if len(df_collision) > 0 and 'strategy' in df_collision.columns:
    print('=== 어간말모음 × 전략 ===')
    ct_vowel = pd.crosstab(df_collision['dict_stem_final_vowel'], df_collision['strategy'], margins=True)
    print(ct_vowel)
    print()
    ct_vowel_pct = pd.crosstab(df_collision['dict_stem_final_vowel'], df_collision['strategy'], normalize='index').round(3)
    print(ct_vowel_pct)

## 8. 결과 저장

In [ ]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# 전체 매칭 결과 저장
if len(df_matched) > 0:
    out1 = f'{RESULT_DIR}/seoul_vowel_collision_all_{timestamp}.csv'
    df_matched.to_csv(out1, index=False, encoding='utf-8-sig')
    print(f'전체 매칭: {out1} ({len(df_matched):,}행)')

# 모음충돌 어미 결합만 저장 (핵심)
if len(df_collision) > 0:
    out2 = f'{RESULT_DIR}/seoul_vowel_collision_suffix_{timestamp}.csv'
    df_collision.to_csv(out2, index=False, encoding='utf-8-sig')
    print(f'충돌어미: {out2} ({len(df_collision):,}행)')

print('\n저장 완료')